# Projet RAG 100% Open Source (Hugging Face)

Pipeline RAG complet, gratuit et local :
- **Données** : le règlement interne `entreprise_data.txt` **+ des articles Wikipédia FR** sur le droit du travail (téléchargés via l'API Wikipédia, mis en cache dans `data/wikipedia/`)
- **Embeddings locaux** : `intfloat/multilingual-e5-small` (multilingue, bien meilleur en français que `all-MiniLM-L6-v2` qui est entraîné sur de l'anglais)
- **Base vectorielle** : `ChromaDB` (via `langchain-chroma`)
- **LLM de génération** : `Qwen/Qwen2.5-1.5B-Instruct` en local (CPU OK), avec son *chat template*

In [1]:
# Installation des dépendances
%pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-chroma langchain-text-splitters chromadb sentence-transformers transformers requests

Note: you may need to restart the kernel to use updated packages.


## 1. Collecte des données
On combine le fichier local et des articles Wikipédia. Les articles sont téléchargés une seule fois puis relus depuis le disque.
Pour changer de sujet, il suffit de modifier la liste `WIKI_TITRES`. Vous pouvez aussi déposer vos propres fichiers `.txt` dans `data/`.

In [2]:
import re
import time
from pathlib import Path
import requests
from langchain_core.documents import Document

DATA_DIR = Path("data")
WIKI_DIR = DATA_DIR / "wikipedia"
WIKI_DIR.mkdir(parents=True, exist_ok=True)

WIKI_TITRES = [
    "Télétravail",
    "Congé de paternité en France",
    "Durée du travail en France",
    "Comité social et économique",
    "Droit du travail en France",
    "Titre-restaurant",
    "Rupture conventionnelle du contrat de travail",
]

def telecharger_wikipedia(titre, langue="fr"):
    """Télécharge le texte brut d'un article Wikipédia (avec cache disque)."""
    fichier = WIKI_DIR / f"{titre.replace('/', '_')}.txt"
    if fichier.exists():
        return fichier.read_text(encoding="utf-8")
    params = dict(action="query", prop="extracts", explaintext=1, redirects=1,
                  titles=titre, format="json", formatversion=2)
    for tentative in range(4):
        r = requests.get(f"https://{langue}.wikipedia.org/w/api.php", params=params,
                         headers={"User-Agent": "rag-notebook-demo/1.0"}, timeout=30)
        if r.status_code == 200 and r.headers.get("content-type", "").startswith("application/json"):
            page = r.json()["query"]["pages"][0]
            if page.get("missing"):
                raise ValueError(f"Article introuvable : {titre}")
            texte = page["extract"]
            fichier.write_text(texte, encoding="utf-8")
            time.sleep(1)  # politesse envers l'API (évite le rate-limit)
            return texte
        time.sleep(2 ** tentative)  # rate-limit -> on attend et on réessaie
    raise RuntimeError(f"Échec du téléchargement : {titre} (HTTP {r.status_code})")

def nettoyer_wiki(texte):
    """Retire les sections parasites (références, liens...) et la syntaxe == Titre ==."""
    texte = re.split(r"\n==+ ?(Notes et références|Notes|Références|Annexes|Voir aussi|Bibliographie|Liens externes|Articles connexes) ?==+", texte)[0]
    texte = re.sub(r"\[(réf\. nécessaire|réf\. souhaitée|Quand \?|Lequel \?|Qui \?)\]", "", texte)
    texte = re.sub(r"\n==+ ?(.*?) ?==+\n", r"\n\n\1\n", texte)   # "== Histoire ==" -> "Histoire"
    texte = re.sub(r"\n{3,}", "\n\n", texte)
    return texte.strip()

documents = []

# 1) Le règlement interne local
texte_local = Path("entreprise_data.txt").read_text(encoding="utf-8")
documents.append(Document(page_content=texte_local, metadata={"source": "Règlement TechCorp (entreprise_data.txt)"}))

# 2) Les articles Wikipédia
for titre in WIKI_TITRES:
    try:
        texte = telecharger_wikipedia(titre)
        documents.append(Document(page_content=nettoyer_wiki(texte), metadata={"source": f"Wikipédia : {titre}"}))
        print(f"✓ {titre} ({len(texte):,} caractères)")
    except Exception as e:
        print(f"✗ {titre} : {e}")

# 3) Vos propres fichiers .txt déposés dans data/ (hors cache Wikipédia)
for f in sorted(DATA_DIR.glob("*.txt")):
    documents.append(Document(page_content=f.read_text(encoding="utf-8"), metadata={"source": f.name}))
    print(f"✓ {f.name}")

print(f"\n{len(documents)} documents chargés.")

✓ Télétravail (53,227 caractères)
✓ Congé de paternité en France (8,727 caractères)
✓ Durée du travail en France (21,662 caractères)
✓ Comité social et économique (19,471 caractères)
✓ Droit du travail en France (34,763 caractères)
✓ Titre-restaurant (5,521 caractères)
✓ Rupture conventionnelle du contrat de travail (22,217 caractères)

8 documents chargés.


## 2. Découpage et indexation
- Articles : chunks de ~800 caractères avec chevauchement, coupés de préférence sur les paragraphes puis les phrases.
- Règlement interne : petits chunks (~300 caractères), une section par chunk. Un seul chunk mélangeant télétravail, congés et machine à café donnait un vecteur « moyen » mal classé.
- Le modèle e5 attend les préfixes `passage: ` (documents) et `query: ` (questions).
- **La collection est vidée avant chaque indexation** : avant, chaque exécution ré-ajoutait les mêmes chunks dans Chroma, d'où les 3 extraits identiques renvoyés par la recherche.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

splitter_articles = RecursiveCharacterTextSplitter(
    chunk_size=800, chunk_overlap=120, separators=["\n\n", "\n", ". ", " ", ""],
)
splitter_reglement = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=0, separators=["\n\n", "\n", ". "],
)

chunks = []
for doc in documents:
    if doc.metadata["source"].startswith("Wikipédia"):
        morceaux = splitter_articles.split_documents([doc])
    else:
        morceaux = splitter_reglement.split_documents([doc])
    # On retire les chunks trop courts (titres de sections isolés)
    chunks += [c for c in morceaux if len(c.page_content.strip()) > 80]
print(f"{len(chunks)} segments créés.")

embedding = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True},
)

persist_directory = "./vectordb_os"
vectordb = Chroma(
    collection_name="rag_entreprise",
    embedding_function=embedding,
    persist_directory=persist_directory,
    collection_metadata={"hnsw:space": "cosine"},
)
vectordb.reset_collection()          # évite les doublons si on relance la cellule
vectordb.add_documents(chunks)
print(f"{vectordb._collection.count()} segments indexés dans {persist_directory}")

292 segments créés.


/home/ensimag/rag/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1459.24it/s]


292 segments indexés dans ./vectordb_os


## 3. Recherche (Retrieval)
On affiche les extraits trouvés avec leur source et leur score de pertinence (1 = identique).

In [5]:
# Pour relancer le notebook plus tard sans réindexer, on peut juste rouvrir la base :
# vectordb = Chroma(collection_name="rag_entreprise", embedding_function=embedding, persist_directory=persist_directory)

K = 4
retriever = vectordb.as_retriever(search_kwargs={"k": K})

question = "c'est quoi la durée du congé de paternité et d'accueil de l'enfant?"
resultats = vectordb.similarity_search_with_relevance_scores(question, k=K)

print(f"--- {len(resultats)} extraits pour : '{question}' ---")
for i, (doc, score) in enumerate(resultats, 1):
    print(f"\n[Extrait {i}] score={score:.3f} | {doc.metadata['source']}")
    print(doc.page_content[:300] + ("..." if len(doc.page_content) > 300 else ""))

--- 4 extraits pour : 'c'est quoi la durée du congé de paternité et d'accueil de l'enfant?' ---

[Extrait 1] score=0.948 | Wikipédia : Congé de paternité en France
En France, le congé de paternité et de l’accueil de l’enfant est un droit dont bénéficie le père salarié ainsi que, le cas échéant, le conjoint ou concubin de la mère ou la personne salariée liée à elle par un pacte civil de solidarité. Le congé paternité est créé en 2002, d’une durée de onze jours....

[Extrait 2] score=0.935 | Wikipédia : Congé de paternité en France
Description
Ce droit existe aussi dans le cadre d'une adoption. Dans le cadre du congé d'adoption, un partage des 10 semaines entre le père et la mère est possible. Dans ce cas, le congé d'adoption est augmenté de 11 jours.
Le congé de paternité ne s'applique qu'aux parents reconnus par l'état civil...

[Extrait 3] score=0.919 | Wikipédia : Congé de paternité en France
Depuis le 1er juillet 2021, sous le gouvernement Castex, la durée du congé de paternité est 

## 4. Génération (Qwen2.5-1.5B-Instruct en local)
Corrections par rapport à la version précédente :
- **`ChatHuggingFace`** applique le *chat template* de Qwen. Avant, `HuggingFacePipeline` recevait un texte brut `System: ... Human: ...`, le modèle ne savait pas où s'arrêter et partait en monologue (« Pourriez-vous m'expliquer… »).
- **Décodage déterministe** (`do_sample=False`) au lieu de `temperature=0.0001`.
- Le contexte est **numéroté avec ses sources** et la réponse les cite.
- **Anti-hallucination** à deux niveaux : (1) si aucun extrait ne dépasse `SEUIL_PERTINENCE`, on refuse sans appeler le LLM ; (2) la consigne de refus est répétée dans le message utilisateur, car un petit modèle suit mieux la dernière instruction (sinon il répondait « Canberra » à une question hors sujet).

In [6]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # "Qwen/Qwen2.5-0.5B-Instruct" si la machine est lente

pipe = pipeline(
    "text-generation",
    model=MODEL_ID,
    max_new_tokens=250,
    do_sample=False,
    repetition_penalty=1.05,
    return_full_text=False,
)
llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=pipe))

REFUS = "Je ne trouve pas cette information dans les documents."
SEUIL_PERTINENCE = 0.79   # propre à multilingual-e5 (scores tassés entre ~0.75 et ~0.95) ; à ajuster si besoin

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es un assistant RH qui répond en français, uniquement à partir des extraits numérotés fournis. "
     "Tu n'utilises jamais tes connaissances générales.\n\n"
     "Extraits :\n{context}"),
    ("human",
     "Question : {question}\n\n"
     "Consignes : réponds en 1 à 4 phrases en citant les extraits utilisés, ex. [1]. "
     "Si les extraits ne contiennent pas la réponse, réponds uniquement : « " + REFUS + " »"),
])

chain = prompt | llm | StrOutputParser()

def formater_contexte(docs):
    return "\n\n".join(f"[{i}] (source : {d.metadata['source']})\n{d.page_content}"
                         for i, d in enumerate(docs, 1))

def repondre(question, k=K, afficher_sources=True):
    resultats = vectordb.similarity_search_with_relevance_scores(question, k=k)
    docs = [d for d, _ in resultats]
    meilleur_score = resultats[0][1] if resultats else 0.0
    if meilleur_score < SEUIL_PERTINENCE:
        reponse, docs = REFUS, []          # rien de pertinent : inutile d'appeler le LLM
    else:
        reponse = chain.invoke({"context": formater_contexte(docs), "question": question}).strip()
    print(f"❓ {question}\n💬 {reponse}")
    if afficher_sources:
        for i, d in enumerate(docs, 1):
            print(f"   [{i}] {d.metadata['source']}")
    print()
    return reponse

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 669.39it/s]


In [7]:
questions = [
    "Combien de jours de télétravail ai-je droit chez TechCorp ?",
    "Qu'est-ce qu'une rupture conventionnelle ?",
    "Quel est le rôle du comité social et économique ?",
    "Combien de jours de congé pour un mariage ?",
    "Quelle est la capitale de l'Australie ?",   # hors sujet -> doit refuser
    "Quel est le prix d'une Tesla ?",            # hors sujet -> doit refuser
]
for q in questions:
    repondre(q)

❓ Combien de jours de télétravail ai-je droit chez TechCorp ?
💬 [1] Les employés de TechCorp ont droit à 3 jours de télétravail par semaine.
   [1] Règlement TechCorp (entreprise_data.txt)
   [2] Wikipédia : Télétravail
   [3] Wikipédia : Télétravail
   [4] Wikipédia : Télétravail

❓ Qu'est-ce qu'une rupture conventionnelle ?
💬 Une rupture conventionnelle est une méthode de séparation amiable entre l'employeur et le salarié lié par un contrat de travail à durée indéterminée (CDI). Elle est introduite par la loi portant modernisation du marché du travail du 25 juin 2008. Cette convention permet aux parties de négocier sur les modalités de départ à la démission et au licenciement. Elle offre l'avantage d'éligibilité à l'indemnisation chômage par France Travail, ce qui n'est pas le cas à l'issue d'une démission. La convention peut être signée le jour de l'entretien ou ultérieurement, avec une date de départ et une indemnité spécifique de rupture librement négociées. Une fois signée, chaqu

---
## 5. Évaluation chiffrée
Les tests ci-dessus montrent un **échec** : la question « Quel est le prix d'une Tesla ? » n'est pas refusée. L'article *Télétravail*
mentionne le patron de Tesla, donc le score dépasse le seuil de 0,79, choisi à la main. Les scores cosinus de e5 sont **tassés**
(~0,75 à 0,95) : un seuil fixé sur quelques exemples est fragile.

On construit donc un **jeu d'évaluation** (15 questions dans le périmètre, avec la source attendue, et 10 hors sujet, dont des pièges)
et on compare trois méthodes de recherche :
- **dense** (e5, celle du notebook) : comprend le sens, mais rate parfois un mot ou un nombre exact ;
- **BM25** : correspondance de mots-clés pondérée par leur rareté ; fort sur les termes exacts, aveugle aux synonymes ;
- **hybride** (*Reciprocal Rank Fusion*) : `RRF(d) = Σ 1 / (60 + rang(d))`, qui combine les deux classements sans calibrer les scores.

Métriques :
- **Hit@1 / Hit@K** : la bonne source est-elle en tête / dans les K premiers passages ?
- **MRR** : moyenne de `1 / rang` de la première bonne source.
- **AUROC de refus** : le meilleur score sépare-t-il les questions pertinentes des questions hors sujet ?
- **Seuil optimal** (exactitude équilibrée) au lieu du seuil choisi à la main.

In [ ]:
# Jeu d'évaluation : (question, source qui contient la réponse ou None si hors sujet, mots-clés attendus)
QUESTIONS = [
    ("Combien de jours de télétravail par semaine ai-je droit chez TechCorp ?", "Règlement TechCorp (entreprise_data.txt)", ["3"]),
    ("Combien de jours de congé pour un mariage chez TechCorp ?", "Règlement TechCorp (entreprise_data.txt)", ["4"]),
    ("Quelle est la durée du congé paternité chez TechCorp ?", "Règlement TechCorp (entreprise_data.txt)", ["28"]),
    ("En quelle année le congé de paternité a-t-il été créé en France ?", "Wikipédia : Congé de paternité en France", ["2002"]),
    ("Depuis quand le congé de paternité dure-t-il vingt-cinq jours ?", "Wikipédia : Congé de paternité en France", ["2021"]),
    ("Qui peut bénéficier du congé de paternité et d'accueil de l'enfant ?", "Wikipédia : Congé de paternité en France", []),
    ("Quel est le délai de rétractation après la signature d'une rupture conventionnelle ?", "Wikipédia : Rupture conventionnelle du contrat de travail", ["quinze", "15"]),
    ("Quelle loi a introduit la rupture conventionnelle ?", "Wikipédia : Rupture conventionnelle du contrat de travail", ["2008"]),
    ("Quelle est la durée légale hebdomadaire du travail en France ?", "Wikipédia : Durée du travail en France", ["35"]),
    ("Quel est le rôle du comité social et économique ?", "Wikipédia : Comité social et économique", []),
    ("Quelles instances ont été fusionnées dans le CSE ?", "Wikipédia : Comité social et économique", []),
    ("Qu'est-ce qu'un titre-restaurant ?", "Wikipédia : Titre-restaurant", []),
    ("Qui finance le titre-restaurant ?", "Wikipédia : Titre-restaurant", []),
    ("Qu'est-ce que le télétravail ?", "Wikipédia : Télétravail", []),
    ("Quelles sont les sources du droit du travail en France ?", "Wikipédia : Droit du travail en France", []),
    ("Quelle est la capitale de l'Australie ?", None, []),
    ("Quel est le prix d'une Tesla ?", None, []),
    ("Quelle est la recette des crêpes ?", None, []),
    ("Qui a gagné la Coupe du monde de football 2018 ?", None, []),
    ("Quelle est la vitesse de la lumière ?", None, []),
    ("Comment entraîner un réseau de neurones ?", None, []),
    ("Quelle est la hauteur de la tour Eiffel ?", None, []),
    ("Qui a écrit Les Misérables ?", None, []),
    ("Quel temps fera-t-il demain à Grenoble ?", None, []),
    ("Combien coûte un abonnement Netflix ?", None, []),
]
print(sum(s is not None for _, s, _ in QUESTIONS), 'questions dans le périmètre,', sum(s is None for _, s, _ in QUESTIONS), 'hors périmètre')

In [ ]:
# !pip install -q rank_bm25 scikit-learn
import re, unicodedata
import numpy as np
import pandas as pd
from rank_bm25 import BM25Okapi
from sklearn.metrics import roc_auc_score

STOP = set("au aux avec ce ces dans de des du elle en et il ils je la le les leur lui ma mais me mes moi mon ne nos notre nous on ou "
           "par pas pour qu que qui sa se ses son sur ta te tes toi ton tu un une vos votre vous c d j l à m n s t y est sont a "
           "quel quelle quels quelles combien quoi comment".split())

def tokenize(text):
    text = unicodedata.normalize("NFKD", text.lower())
    text = "".join(ch for ch in text if not unicodedata.combining(ch))     # sans accents
    return [t for t in re.findall(r"[a-z0-9]+", text) if t not in STOP and len(t) > 1]

bm25 = BM25Okapi([tokenize(c.page_content) for c in chunks])

def search_dense(q, k):
    return vectordb.similarity_search_with_relevance_scores(q, k=k)

def search_bm25(q, k):
    scores = bm25.get_scores(tokenize(q))
    best = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in best]

def search_hybrid(q, k, depth=20):
    fused, docs = {}, {}
    for ranking in (search_dense(q, depth), search_bm25(q, depth)):
        for rank, (doc, _) in enumerate(ranking, start=1):
            key = doc.page_content
            fused[key] = fused.get(key, 0.0) + 1.0 / (60 + rank)
            docs[key] = doc
    best = sorted(fused, key=fused.get, reverse=True)[:k]
    return [(docs[key], fused[key]) for key in best]

def balanced_accuracy(in_scope, scores, t):
    answer = scores >= t
    return (answer[in_scope].mean() + (~answer[~in_scope]).mean()) / 2

def evaluate_retriever(search, k=K, given_threshold=None):
    ranks, best_scores, in_scope = [], [], []
    for q, source, _ in QUESTIONS:
        hits = search(q, k)
        best_scores.append(hits[0][1] if hits else -np.inf)
        in_scope.append(source is not None)
        if source is not None:
            sources = [d.metadata["source"] for d, _ in hits]
            ranks.append(sources.index(source) + 1 if source in sources else None)
    s, ins = np.array(best_scores), np.array(in_scope)
    best_t = max(np.unique(s), key=lambda t: balanced_accuracy(ins, s, t))
    out = {"Hit@1": np.mean([r == 1 for r in ranks]), f"Hit@{k}": np.mean([r is not None for r in ranks]),
           "MRR": np.mean([1 / r if r else 0 for r in ranks]), "AUROC refus": roc_auc_score(ins, s),
           "seuil optimal": best_t, "exactitude équilibrée": balanced_accuracy(ins, s, best_t)}
    if given_threshold is not None:
        out["exactitude au seuil du notebook"] = balanced_accuracy(ins, s, given_threshold)
    return out

comparaison = pd.DataFrame({
    "dense (e5)": evaluate_retriever(search_dense, given_threshold=SEUIL_PERTINENCE),
    "BM25": evaluate_retriever(search_bm25),
    "hybride (RRF)": evaluate_retriever(search_hybrid),
}).T.round(3)
comparaison

In [ ]:
# Le cas Tesla : quels scores obtient-il avec chaque méthode, comparé aux questions pertinentes ?
for name, search in [("dense (e5)", search_dense), ("BM25", search_bm25), ("hybride (RRF)", search_hybrid)]:
    s_in = [search(q, 1)[0][1] for q, src, _ in QUESTIONS if src is not None]
    s_tesla = search("Quel est le prix d'une Tesla ?", 1)[0][1]
    seuil = comparaison.loc[name, "seuil optimal"]
    print(f"{name:14s} Tesla = {s_tesla:.3f} | questions pertinentes : min {min(s_in):.3f}, médiane {np.median(s_in):.3f} "
          f"| seuil optimal {seuil:.3f} -> {'refusée ✅' if s_tesla < seuil else 'NON refusée ❌'}")

### Évaluation de bout en bout (avec le LLM, plus lent)
On relance l'assistant complet (`repondre`) sur toutes les questions, avec le **seuil optimal** de la recherche dense :
- questions pertinentes : la réponse ne doit pas être un refus, et doit contenir le mot-clé attendu quand il y en a un ;
- questions hors sujet : la réponse doit être le refus.

In [ ]:
SEUIL_PERTINENCE = float(comparaison.loc["dense (e5)", "seuil optimal"])
lignes = []
for q, source, mots in QUESTIONS:
    reponse = repondre(q, afficher_sources=False)
    refus = REFUS in reponse
    if source is None:
        correct = refus
    else:
        correct = (not refus) and (not mots or any(m.lower() in reponse.lower() for m in mots))
    lignes.append({"question": q, "dans le périmètre": source is not None, "refus": refus, "correct": correct})
bilan = pd.DataFrame(lignes)
dans = bilan[bilan["dans le périmètre"]]
hors = bilan[~bilan["dans le périmètre"]]
print(f"Questions pertinentes bien traitées : {dans.correct.mean():.0%} | faux refus : {dans.refus.mean():.0%} "
      f"| questions hors sujet refusées : {hors.refus.mean():.0%}")
bilan

### Pistes d'amélioration
- **Re-classement par cross-encoder** : relire les 20 meilleurs passages en lisant question et passage ensemble ; son score est
  aussi un meilleur signal de refus.
- Vérifier la **fidélité** des réponses (affirmations soutenues par les extraits ?), par exemple avec un LLM juge.
- Gérer les **sources contradictoires** (Wikipédia : 25 jours de congé de paternité ; règlement TechCorp : 28 jours ouvrés).
- Interface de discussion (Gradio / Streamlit) et API (FastAPI) ; modèle plus grand quantifié si GPU.